# Table 2: Spike yield and classifier performance

Problems (see `../../../submission/PROBLEMS.md`):

- The current sorted input lacks `EXP106_AD-CO-DAY150-30_NOV24`. To reproduce the submitted 50-session summary, this notebook represents that missing sorted session as zero valid sorted units.
- Double rounding: I originally reported 3 decimal places with rounding, then Fan rounded again to 2 decimal places, which messed up Unsorted block AUC: it was `0.954570...` and I reported `0.955` which Fan rounded to `0.96`, but if we immediately rounded to 2 places, it would become `0.95`

In [3]:
from manuscript_analysis import find_project_root
from decimal import Decimal, ROUND_HALF_UP
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from manuscript_analysis.figures.export import save_label_variants


ROOT = find_project_root()
ANALYSIS_DIR = ROOT / 'results' / 'matched_three_pipeline_run' / 'analysis_exports'
OUTPUT_PATH = ROOT / 'output' / 'tables' / '2' / '2_spike_yield.csv'
FIGURE_STEM = '2_spike_yield'

MASTER_COHORT_PATH = ANALYSIS_DIR / 'session_manifest_master_50.csv'
SPIKE_COUNTS_PATH = ANALYSIS_DIR / 'spike_counts_long.csv'
RUN_METRICS_PATH = ANALYSIS_DIR / 'run_metrics.csv'

MISSING_SORTED_SESSION = 'exp106_ad-co-day150-30_nov24'
PIPELINES = ('SORTED', 'PRUNED', 'UNSORTED')
DISPLAY_NAMES = {'SORTED': 'Sorted', 'PRUNED': 'Pruned', 'UNSORTED': 'Unsorted'}


In [4]:
master = pd.read_csv(MASTER_COHORT_PATH)
spike_counts = pd.read_csv(SPIKE_COUNTS_PATH)
run_metrics = pd.read_csv(RUN_METRICS_PATH)

cohort = master['session_key'].tolist()
if len(cohort) != 50 or len(set(cohort)) != 50:
    raise ValueError('Table 2 requires exactly 50 unique master-cohort sessions.')

raw_counts = (
    spike_counts.loc[spike_counts['session_key'].isin(cohort)]
    .pivot(index='session_key', columns='pipeline', values='spike_count_raw')
    .reindex(index=cohort, columns=PIPELINES)
)

missing_sorted = set(raw_counts.index[raw_counts['SORTED'].isna()])
if missing_sorted - {MISSING_SORTED_SESSION}:
    raise ValueError(f'Unexpected missing sorted sessions: {sorted(missing_sorted)}')

raw_counts.loc[MISSING_SORTED_SESSION, 'SORTED'] = 0

if raw_counts.isna().any().any():
    raise ValueError('Every Table 2 count must be present after the documented recovery step.')

raw_counts.describe().T[['count', '50%', '25%', '75%']]


,count,50%,25%,75%
pipeline,,,,
SORTED,50.0,25057.0,5260.00,55899.50
PRUNED,50.0,70346.0,20796.75,200152.25
UNSORTED,50.0,90027.5,24128.75,340918.50


In [6]:
def publication_count(value: float) -> int:
    return int(round(value))


def iqr_text(values: pd.Series) -> str:
    lower = round(values.quantile(0.25), 2)
    upper = round(values.quantile(0.75), 2)
    return f'[{lower}, {upper}]'


rows = []
for pipeline in PIPELINES:
    metrics = run_metrics.loc[run_metrics['pipeline'].eq(pipeline)]
    if len(metrics) != 25:
        raise ValueError(f'{pipeline} must have 25 repeated runs, found {len(metrics)}.')
    rows.append({
        'approach': DISPLAY_NAMES[pipeline],
        'count_median': int(round((raw_counts[pipeline].quantile(0.50)))),
        'count_iqr': iqr_text(raw_counts[pipeline]),
        'block_auc_median': round(metrics['block_auc'].quantile(0.50), 2),
        'block_auc_iqr': iqr_text(metrics['block_auc']),
        'session_auc_median': round(metrics['session_auc'].quantile(0.50), 2),
        'session_auc_iqr': iqr_text(metrics['session_auc']),
    })

table_2 = pd.DataFrame(rows)
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
table_2.to_csv(OUTPUT_PATH, index=False)

fig, ax = plt.subplots(figsize=(11.5, 1.6))
ax.axis('off')
rendered_table = ax.table(
    cellText=table_2.to_numpy(),
    colLabels=table_2.columns,
    loc='center',
    cellLoc='center',
)
rendered_table.auto_set_font_size(False)
rendered_table.set_fontsize(8)
rendered_table.scale(1, 1.45)
saved_paths = save_label_variants(
    fig, OUTPUT_PATH.parent, FIGURE_STEM, bbox_inches='tight', facecolor='white'
)

print(f'Wrote {OUTPUT_PATH}')
for saved_path in saved_paths.values():
    print(f'Saved {saved_path.relative_to(ROOT)}')
table_2

Wrote C:\Users\cchst\OneDrive\Desktop\DBC\manuscript-analysis\output\tables\2\2_spike_yield.csv


,approach,count_median,count_iqr,block_auc_median,block_auc_iqr,session_auc_median,session_auc_iqr
0,Sorted,25057,"[5260.0, 55899.5]",0.85,"[0.69, 0.88]",0.93,"[0.8, 1.0]"
1,Pruned,70346,"[20796.75, 200152.25]",0.89,"[0.82, 0.95]",0.96,"[0.93, 0.96]"
2,Unsorted,90028,"[24128.75, 340918.5]",0.91,"[0.86, 0.95]",0.96,"[0.93, 1.0]"


## Line 409 test:

paired two-tailed Wilcoxon signed-rank tests for significant difference between spike counts for sorted, unsorted, and pruned; all significant `(p < 0.001)`

In [7]:
from scipy.stats import wilcoxon

pairs = [
    ("UNSORTED", "PRUNED"),
    ("UNSORTED", "SORTED"),
    ("PRUNED", "SORTED"),
]

wilcoxon_results = pd.DataFrame(
    [
        {
            "comparison": f"{left.title()} vs {right.title()}",
            "n_pairs": len(raw_counts),
            "W": test.statistic,
            "p_two_sided": test.pvalue,
        }
        for left, right in pairs
        for test in [
            wilcoxon(
                raw_counts[left],
                raw_counts[right],
                alternative="two-sided",
                method="auto",
            )
        ]
    ]
)

wilcoxon_results

,comparison,n_pairs,W,p_two_sided
0,Unsorted vs Pruned,50,0.0,1.776357e-15
1,Unsorted vs Sorted,50,63.0,2.522675e-10
2,Pruned vs Sorted,50,79.0,1.314969e-09


## Line 419-422 tests: block-level AUC comparisons

two-tailed Bonferroni corrected Wilcoxon signed-rank tests for differences between sorted, unsorted, and pruned auc median across 25 CV splits; pruned and unsorted significantly better than sorted, unsorted and pruned comparable

In [9]:
auc_by_seed = run_metrics.pivot(
    index="seed",
    columns="pipeline",
    values="block_auc",
)

comparisons = [
    ("UNSORTED", "SORTED"),
    ("UNSORTED", "PRUNED"),
    ("PRUNED", "SORTED"),
]

p_value_rows = []
for first, second in comparisons:
    paired = auc_by_seed[[first, second]].dropna()

    result = wilcoxon(
        paired[first],
        paired[second],
        alternative="two-sided",
        zero_method="wilcox",
    )
    raw_p = result.pvalue

    p_value_rows.append(
        {
            "comparison": f"{first.title()} vs. {second.title()}",
            "n_pairs": len(paired),
            "p_raw": raw_p,
            "p_bonferroni_3_tests": min(1.0, raw_p * len(comparisons)),
        }
    )

p_values = pd.DataFrame(p_value_rows)
p_values

,comparison,n_pairs,p_raw,p_bonferroni_3_tests
0,Unsorted vs. Sorted,25,0.000287,0.000861
1,Unsorted vs. Pruned,25,0.070984,0.212951
2,Pruned vs. Sorted,25,0.000631,0.001894
